In [1]:
import pandas as pd
import re

Load the excel file into a DataFrame.
Specify the target sheet, header row, number of final rows to skip, and columns to use.

In [2]:
df_raw = pd.read_excel('.raw/region_population_area_density.xlsx', sheet_name='T1', header=1, skipfooter=5, usecols="B:E")
df_raw

,Region,Population,Land Area (Square Kilometer),Population Density (Persons Per Square Kilometer of land)
0,Philippines,"112,727,776 a","300,000.00 b",376
1,National Capital Region (NCR),14001751,620.61,22561
2,Region IV-A (CALABARZON),16933234,15371.44,1102
3,Region VII (Central Visayas),6640875,8975.04,740
4,Region III (Central Luzon),12989074,22648.03,574
5,Region I (Ilocos Region),5342453,12830.62,416
6,Region VI (Western Visayas),4861911,12641.59,385
7,Negros Island Region (NIR),4904944,13553.48,362
8,Region V (Bicol Region),6064426,18007.22,337
9,Bangsamoro Autonomous Region in Muslim Mindana...,4545486,14478.08,314


Rename the columns

In [3]:
df_raw.columns = ['region', 'population', "land_area", "population_density"]
df_raw

,region,population,land_area,population_density
0,Philippines,"112,727,776 a","300,000.00 b",376
1,National Capital Region (NCR),14001751,620.61,22561
2,Region IV-A (CALABARZON),16933234,15371.44,1102
3,Region VII (Central Visayas),6640875,8975.04,740
4,Region III (Central Luzon),12989074,22648.03,574
5,Region I (Ilocos Region),5342453,12830.62,416
6,Region VI (Western Visayas),4861911,12641.59,385
7,Negros Island Region (NIR),4904944,13553.48,362
8,Region V (Bicol Region),6064426,18007.22,337
9,Bangsamoro Autonomous Region in Muslim Mindana...,4545486,14478.08,314


Clean the data

In [4]:
df_raw['population'] = pd.to_numeric(
    df_raw['population'].replace(r'[^\d.]', '', regex=True), 
    errors='coerce'
)
df_raw['land_area'] = pd.to_numeric(
    df_raw['land_area'].replace(r'[^\d.]', '', regex=True), 
    errors='coerce'
)
df_raw

,region,population,land_area,population_density
0,Philippines,112727776,300000.00,376
1,National Capital Region (NCR),14001751,620.61,22561
2,Region IV-A (CALABARZON),16933234,15371.44,1102
3,Region VII (Central Visayas),6640875,8975.04,740
4,Region III (Central Luzon),12989074,22648.03,574
5,Region I (Ilocos Region),5342453,12830.62,416
6,Region VI (Western Visayas),4861911,12641.59,385
7,Negros Island Region (NIR),4904944,13553.48,362
8,Region V (Bicol Region),6064426,18007.22,337
9,Bangsamoro Autonomous Region in Muslim Mindana...,4545486,14478.08,314


Import csv template file. Match raw data with template.

In [5]:
df_csv = pd.read_csv(
    'templates/regions.csv',
    dtype={
        'iso_code': 'str',
        'psgc_code': 'str',
        'psgc10_code': 'str',
        'region_name': 'str',
    }
)
df_csv

,iso_code,psgc_code,psgc10_code,region_name
0,01,01,0100000000,REGION I (ILOCOS REGION)
1,15,14,1400000000,CORDILLERA ADMINISTRATIVE REGION (CAR)
2,02,02,0200000000,REGION II (CAGAYAN VALLEY)
3,03,03,0300000000,REGION III (CENTRAL LUZON)
4,00,13,1300000000,NATIONAL CAPITAL REGION (NCR)
5,40,04,0400000000,REGION IV-A (CALABARZON)
6,41,17,1700000000,REGION IV-B (MIMAROPA)
7,05,05,0500000000,REGION V (BICOL REGION)
8,06,06,0600000000,REGION VI (WESTERN VISAYAS)
9,16,18,1800000000,NEGROS ISLAND REGION (NIR)


In [6]:
region_names = list(df_csv['region_name'])
region_names

['REGION I (ILOCOS REGION)',
 'CORDILLERA ADMINISTRATIVE REGION (CAR)',
 'REGION II (CAGAYAN VALLEY)',
 'REGION III (CENTRAL LUZON)',
 'NATIONAL CAPITAL REGION (NCR)',
 'REGION IV-A (CALABARZON)',
 'REGION IV-B (MIMAROPA)',
 'REGION V (BICOL REGION)',
 'REGION VI (WESTERN VISAYAS)',
 'NEGROS ISLAND REGION (NIR)',
 'REGION VII (CENTRAL VISAYAS)',
 'REGION VIII (EASTERN VISAYAS)',
 'REGION IX (ZAMBOANGA PENINSULA)',
 'REGION X (NORTHERN MINDANAO)',
 'REGION XIII (CARAGA)',
 'BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANAO (BARMM)',
 'REGION XII (SOCCSKSARGEN)',
 'REGION XI (DAVAO REGION)']

In [7]:
def rename_region(region):
    for region_name in region_names:
        first_two_words = " ".join(region_name.split(maxsplit=2)[:2]) + " "
        between_parentheses = re.search(r"\((.*?)\)", region_name)
        between_parentheses = between_parentheses.group(1) + " " if between_parentheses else "xxxx"
        if first_two_words.casefold() in region.casefold() or between_parentheses.casefold() in region.casefold():
            return region_name
    return "Unmatched: " + region

df_raw['region'] = df_raw['region'].apply(rename_region)
df_raw

,region,population,land_area,population_density
0,Unmatched: Philippines,112727776,300000.00,376
1,NATIONAL CAPITAL REGION (NCR),14001751,620.61,22561
2,REGION IV-A (CALABARZON),16933234,15371.44,1102
3,REGION VII (CENTRAL VISAYAS),6640875,8975.04,740
4,REGION III (CENTRAL LUZON),12989074,22648.03,574
5,REGION I (ILOCOS REGION),5342453,12830.62,416
6,REGION VI (WESTERN VISAYAS),4861911,12641.59,385
7,NEGROS ISLAND REGION (NIR),4904944,13553.48,362
8,REGION V (BICOL REGION),6064426,18007.22,337
9,BANGSAMORO AUTONOMOUS REGION IN MUSLIM MINDANA...,4545486,14478.08,314


In [8]:
df_result = pd.merge(df_csv, df_raw, left_on='region_name', right_on='region', how="left").drop(columns=['region'])
df_result

,iso_code,psgc_code,psgc10_code,region_name,population,land_area,population_density
0,01,01,0100000000,REGION I (ILOCOS REGION),5342453,12830.62,416
1,15,14,1400000000,CORDILLERA ADMINISTRATIVE REGION (CAR),1808985,21116.05,86
2,02,02,0200000000,REGION II (CAGAYAN VALLEY),3777608,30986.61,122
3,03,03,0300000000,REGION III (CENTRAL LUZON),12989074,22648.03,574
4,00,13,1300000000,NATIONAL CAPITAL REGION (NCR),14001751,620.61,22561
5,40,04,0400000000,REGION IV-A (CALABARZON),16933234,15371.44,1102
6,41,17,1700000000,REGION IV-B (MIMAROPA),3245446,29063.47,112
7,05,05,0500000000,REGION V (BICOL REGION),6064426,18007.22,337
8,06,06,0600000000,REGION VI (WESTERN VISAYAS),4861911,12641.59,385
9,16,18,1800000000,NEGROS ISLAND REGION (NIR),4904944,13553.48,362


Export CSV file to designated folder

In [9]:
df_result.to_csv('data/region/population_2024.csv', index=False)